# 07 - 区分線形 PLS（繊維飽和点で分割）

## 動機
木材中の水には「結合水（<30%）」と「自由水（>30%）」の2状態があり、
繊維飽和点（~30%）を境にNIRスペクトルの挙動が変わる（PCA PC1 の非線形性の原因）。

## 戦略
```
model_low   : y < 30% のサンプルでPLS学習（結合水ドメイン）
model_high  : y >= 30% のサンプルでPLS学習（自由水ドメイン）
model_router: PLS3（全サンプル）→ 仮予測で30%以上/以下を判定
```

⚠️ THRESHOLD=30 は物理的根拠による固定値。CVで最適化しない。

In [1]:
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error
from pathlib import Path

Path('../submissions').mkdir(parents=True, exist_ok=True)

def snv(X):
    return (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

print('OK')

OK


In [2]:
train = pd.read_csv('../data/train_near.csv', encoding='cp932')
test  = pd.read_csv('../data/test_near.csv',  encoding='cp932')

META_COLS = ['sample number', 'species number', '樹種', '含水率']
SPEC_COLS = [c for c in train.columns if c not in META_COLS]

X_train = train[SPEC_COLS].values
X_test  = test[SPEC_COLS].values
y_train = train['含水率'].values
groups  = train['樹種'].values
y_min, y_max = y_train.min(), y_train.max()

X_train_snv = snv(X_train)
X_test_snv  = snv(X_test)

THRESHOLD = 30.0  # 繊維飽和点（物理的固定値）

low_ratio = (y_train < THRESHOLD).mean()
print(f'Train: {X_train.shape}')
print(f'y < 30% : {low_ratio:.1%}  ({(y_train < THRESHOLD).sum()}件)')
print(f'y >= 30%: {1-low_ratio:.1%}  ({(y_train >= THRESHOLD).sum()}件)')

Train: (1322, 1555)
y < 30% : 51.1%  (675件)
y >= 30%: 48.9%  (647件)


In [3]:
def fit_piecewise(X, y, n_low=2, n_high=2, n_router=3, threshold=30.0):
    """
    3モデルをfitして返す。
    router: 全サンプルのPLS（仮予測用）
    low   : y < threshold のPLS
    high  : y >= threshold のPLS
    """
    router = PLSRegression(n_components=n_router)
    router.fit(X, y)

    mask_low  = y < threshold
    mask_high = y >= threshold

    pls_low  = PLSRegression(n_components=n_low)
    pls_high = PLSRegression(n_components=n_high)
    pls_low.fit(X[mask_low],  y[mask_low])
    pls_high.fit(X[mask_high], y[mask_high])

    return router, pls_low, pls_high


def predict_piecewise(X, router, pls_low, pls_high, threshold=30.0):
    """
    routerで仮予測 → 30%で振り分けて本予測。
    """
    y_route = router.predict(X).ravel()
    pred    = np.empty(len(X))
    mask_lo = y_route < threshold
    mask_hi = ~mask_lo
    if mask_lo.any():
        pred[mask_lo] = pls_low.predict(X[mask_lo]).ravel()
    if mask_hi.any():
        pred[mask_hi] = pls_high.predict(X[mask_hi]).ravel()
    return pred


print('関数定義完了')

関数定義完了


In [4]:
# LOSO CV: ベースライン vs 区分線形
print('樹種               baseline  piecewise   diff')
print('---------------    --------  ---------  -----')

scores_base = {}
scores_pw   = {}

for sp in np.unique(groups):
    val_mask = groups == sp
    tr_mask  = ~val_mask
    X_tr, X_val = X_train_snv[tr_mask], X_train_snv[val_mask]
    y_tr, y_val = y_train[tr_mask], y_train[val_mask]

    # ベースライン（PLS3）
    pls3 = PLSRegression(n_components=3)
    pls3.fit(X_tr, y_tr)
    s_base = rmse(y_val, pls3.predict(X_val).ravel())

    # 区分線形
    # low/high それぞれのサンプルがあるか確認
    if (y_tr < THRESHOLD).sum() < 3 or (y_tr >= THRESHOLD).sum() < 3:
        s_pw = s_base  # サンプル不足でスキップ
    else:
        router, pls_low, pls_high = fit_piecewise(X_tr, y_tr)
        pred_pw = predict_piecewise(X_val, router, pls_low, pls_high)
        s_pw = rmse(y_val, pred_pw)

    scores_base[sp] = s_base
    scores_pw[sp]   = s_pw
    diff = s_pw - s_base
    mark = '↑改善' if diff < -0.5 else ('↓悪化' if diff > 0.5 else '  ±同')
    print(f'{sp:<18} {s_base:8.2f}  {s_pw:9.2f}  {diff:+6.2f} {mark}')

s_b = pd.Series(scores_base)
s_p = pd.Series(scores_pw)
print(f'\nbaseline   mean: {s_b.mean():.4f}  median: {s_b.median():.4f}')
print(f'piecewise  mean: {s_p.mean():.4f}  median: {s_p.median():.4f}')

樹種               baseline  piecewise   diff
---------------    --------  ---------  -----
イチョウ                  18.20      19.02   +0.81 ↓悪化
ウエンジ                  34.62      31.24   -3.38 ↑改善
ウォールナット               14.19      31.28  +17.09 ↓悪化
クリ                    10.83      10.32   -0.51 ↑改善
スプルース                  5.05       8.42   +3.36 ↓悪化
チェリー                  35.90      38.08   +2.18 ↓悪化
トチ                    12.70      13.75   +1.04 ↓悪化
ナラ                    23.21      33.28  +10.06 ↓悪化
ヒノキ                   15.07      13.76   -1.31 ↑改善
ベイスギ                  71.33      69.39   -1.94 ↑改善
ベイマツ                   6.51       9.08   +2.57 ↓悪化
ホワイトオーク               15.55      22.38   +6.83 ↓悪化
米ヒバ                   14.41      14.45   +0.04   ±同

baseline   mean: 21.3526  median: 15.0709
piecewise  mean: 24.1876  median: 19.0174


In [5]:
# LOSOが改善していれば提出
router, pls_low, pls_high = fit_piecewise(X_train_snv, y_train)
y_pred = predict_piecewise(X_test_snv, router, pls_low, pls_high)
y_pred_clipped = np.clip(y_pred, y_min, y_max)

print(f'予測値（clip前）: min={y_pred.min():.2f}, max={y_pred.max():.2f}')
print(f'予測値（clip後）: min={y_pred_clipped.min():.2f}, max={y_pred_clipped.max():.2f}')

fname = '../submissions/submission_piecewise_pls.csv'
pd.DataFrame({
    'sample number': test['sample number'],
    '含水率': y_pred_clipped
}).to_csv(fname, index=False, header=False)
print(f'\n保存完了: {fname}')

予測値（clip前）: min=4.77, max=171.47
予測値（clip後）: min=4.77, max=171.47

保存完了: ../submissions/submission_piecewise_pls.csv
